# RL velocity lab
Where does the time go when a small model learns with reinforcement learning? This notebook runs Sebastian Raschka's GRPO script (https://github.com/rasbt/reasoning-from-scratch, Apache 2.0) with a stopwatch on five stages: generation, scoring, logprob (the forward pass that scores the rollouts), backward and optimizer. The result is one CSV file, one row per step. The Makefile chooses its name and passes it in, so the file is written under the same name that `make` downloads. Every row also carries the settings, the GPU and the versions, so the file stands alone and files from different runs can be joined. `make results/<name>.csv` sends this notebook to a Colab GPU and downloads that file. The first run downloads the 1.4 GB model.

In [ ]:
import os
SCRIPT = os.environ.get('SCRIPT', 'batched')   # basic: rollouts one after another. batched: in batches
STEPS = int(os.environ.get('STEPS', 10))
ROLLOUTS = int(os.environ.get('ROLLOUTS', 4))
TOKENS = int(os.environ.get('TOKENS', 256))    # most new tokens per rollout
BATCH = int(os.environ.get('BATCH', 2))        # batched script only
SEED = int(os.environ.get('SEED', 42))
SKIP_ZERO_ADVANTAGE = os.environ.get('SKIP_ZERO_ADVANTAGE', '').lower() in ('1', 'true', 'yes')
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/rasbt/reasoning-from-scratch.git')
REPO_REF = os.environ.get('REPO_REF', '2e5e94d47af421d02236f0c28c3525592ceff6c8')   # a full commit hash
NOTEBOOK = os.environ.get('NOTEBOOK', '')      # hash of this notebook, set by the Makefile, so a row names its code
RUN = os.environ.get('RUN', '')                # timestamp of the make run, so rows from several files stay apart
CSV_NAME = os.environ.get('CSV', 'stage_times.csv')   # file name chosen by the Makefile, written to /content/<name>


In [ ]:
import csv, importlib.util, inspect, os, statistics, subprocess, sys, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'No GPU. Colab: Runtime, Change runtime type, choose a GPU.'
GPU = torch.cuda.get_device_name().replace(' ', '-')
os.chdir('/content')
REPO = Path('/content/reasoning-from-scratch')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', REPO_REF], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'sympy', 'requests'], check=True)

# the settings and versions that every CSV row repeats
META = dict(script=SCRIPT, batch=BATCH, rollouts=ROLLOUTS, tokens=TOKENS, seed=SEED, skip_zero=SKIP_ZERO_ADVANTAGE,
            gpu=GPU, torch=torch.__version__, cuda=torch.version.cuda, repo_commit=REPO_REF[:7], notebook=NOTEBOOK[:7], run=RUN)
print(META)


In [ ]:
# Load Raschka's script as a module and wrap five of its functions with timers. Each timer synchronizes
# the GPU first, because GPU work runs in the background and would land in the wrong stage.
STAGES = ['generation', 'scoring', 'logprob', 'backward', 'optimizer']
sys.path.insert(0, str(REPO))
name = 'rlvr_grpo_original_no_kl' + ('_batched' if SCRIPT == 'batched' else '') + '.py'
spec = importlib.util.spec_from_file_location('grpo_script', REPO / 'ch06/02_rlvr_grpo_scripts_intro' / name)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

totals = dict.fromkeys(STAGES, 0.0)

def timed(fn, stage):
    def inner(*a, **k):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        try:
            return fn(*a, **k)
        finally:
            torch.cuda.synchronize()
            totals[stage] += time.perf_counter() - t0
    return inner

sampler = 'sample_response' if hasattr(mod, 'sample_response') else 'sample_responses_batched'
setattr(mod, sampler, timed(getattr(mod, sampler), 'generation'))
mod.reward_rlvr = timed(mod.reward_rlvr, 'scoring')
mod.sequence_logprob = timed(mod.sequence_logprob, 'logprob')
torch.Tensor.backward = timed(torch.Tensor.backward, 'backward')
torch.optim.AdamW.step = timed(torch.optim.AdamW.step, 'optimizer')

CSV = Path('/content') / CSV_NAME
fields = [*META, 'step', *[f'{s}_s' for s in STAGES], 'timed_s', 'wall_s', 'other_s', 'tokens_per_sec', 'avg_response_len', 'reward_avg', 'loss']
handle = open(CSV, 'w', newline='')
writer = csv.DictWriter(handle, fieldnames=fields)
writer.writeheader()
last = {'t': time.perf_counter()}
original_metrics = mod.append_step_metrics

def on_step(step_idx, total_steps, loss, reward_avg, tokens_per_sec, avg_response_len, eval_acc=None):
    wall = time.perf_counter() - last['t']
    row = dict(META)
    row.update({f'{s}_s': round(totals[s], 4) for s in STAGES})
    timed_total = sum(totals.values())
    row.update(step=step_idx, timed_s=round(timed_total, 4), wall_s=round(wall, 4), other_s=round(wall - timed_total, 4),
               tokens_per_sec=round(tokens_per_sec, 2), avg_response_len=round(avg_response_len, 2),
               reward_avg=round(reward_avg, 4), loss=round(loss, 5))
    writer.writerow(row)
    handle.flush()
    for s in STAGES:
        totals[s] = 0.0
    last['t'] = time.perf_counter()
    return original_metrics(step_idx, total_steps, loss, reward_avg, tokens_per_sec, avg_response_len, eval_acc=eval_acc)

mod.append_step_metrics = on_step


In [ ]:
def report():
    rows = list(csv.DictReader(open(CSV)))
    if not rows:
        return print('no step finished')
    use = rows[1:] if len(rows) > 2 else rows   # step 1 pays for warm-up
    mean = {s: statistics.mean(float(r[f'{s}_s']) for r in use) for s in STAGES + ['other']}
    wall = statistics.mean(float(r['wall_s']) for r in use)
    tok = statistics.mean(float(r['tokens_per_sec']) for r in use)
    print(f'{len(rows)} steps, {wall:.1f} s per step, {tok:.1f} tokens per second, peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.2f} GB')
    for s, v in mean.items():
        print(f'{s:11} {v:6.2f} s  {100 * max(v, 0) / wall:5.1f} %')
    import matplotlib.pyplot as plt
    bottom = [0.0] * len(rows)
    for s in STAGES:
        vals = [float(r[f'{s}_s']) for r in rows]
        plt.bar([int(r['step']) for r in rows], vals, bottom=bottom, label=s)
        bottom = [b + v for b, v in zip(bottom, vals)]
    plt.xlabel('step')
    plt.ylabel('seconds')
    plt.legend()
    plt.show()


In [ ]:
from reasoning_from_scratch.ch03 import load_model_and_tokenizer
from reasoning_from_scratch.ch06 import load_math_train

torch.manual_seed(SEED)
device = torch.device('cuda')
model, tokenizer = load_model_and_tokenizer(which_model='base', device=device, use_compile=False)
wanted = dict(model=model, tokenizer=tokenizer, math_data=load_math_train(), math500_eval_data=None, device=device,
              steps=STEPS, num_rollouts=ROLLOUTS, max_new_tokens=TOKENS, temperature=0.8, top_p=0.9, batch_size=BATCH,
              checkpoint_every=0, checkpoint_dir=Path('/content/checkpoints'), eval_max_items=0,
              skip_zero_advantage_updates=SKIP_ZERO_ADVANTAGE, show_eta=True)
accepted = inspect.signature(mod.train_rlvr_grpo).parameters
try:
    mod.train_rlvr_grpo(**{k: v for k, v in wanted.items() if k in accepted})
finally:   # an error such as out of memory still leaves the steps written so far in the CSV file
    handle.close()
    report()
